# 03 — Climate attribution of flood hazard

**Goal**: Quantify how much climate change increased (or decreased) the TC Idai flood by comparing the **factual** scenario against **counterfactual** scenarios — re-runs with climate drivers adjusted to a pre-industrial baseline.

**Prerequisite**: Multiple SFINCS runs have been completed, including the factual run and at least one counterfactual.  
**No simulation is run here** — this notebook only reads existing `sfincs_output_hmax_AllTime.tif` files.

---

## Climate attribution — the concept

In *event attribution*, we compare the **observed world** (factual, current climate) with a **counterfactual world** (what would have happened without anthropogenic climate change). For a tropical cyclone flood like TC Idai, the main climate drivers we adjust are:

| Driver | Factual | Counterfactual example |
|--------|---------|------------------------|
| Precipitation intensity | Current climate ERA5 | −8% (Clausius-Clapeyron scaling with −1°C cooling) |
| Sea level | Current MSL | −0.1 m (pre-industrial SLR subtracted) |
| Wind intensity | Current IBTrACS track | −5% or −10% intensity |

The **counterfactual flood is expected to be smaller** than the factual. The difference gives the climate change contribution.

### Scenario folder naming convention

```
event_tp_{precip}_{CF_rain}_{tide}_{CF_SLR}_{wind}_{CF_wind}
```

- `CF0` = no adjustment (factual climate)
- `CF-8` on the precipitation term = precipitation reduced by 8%
- `CF-0.1` on the SLR term = sea level reduced by 0.1 m
- Multiple drivers can be adjusted simultaneously (compound counterfactual)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rioxarray as rxr
import geopandas as gpd
import contextily as ctx
from pathlib import Path

In [ ]:
# ── Edit these for your event ────────────────────────────────────────────────
BASE   = Path("/p/11210471-001-compass/03_Runs")
REGION = "sofala"
EVENT  = "Idai"

# Define the scenarios to compare as {display_label: scenario_folder_name}
# The first entry is treated as the factual (reference) scenario.
SCENARIOS = {
    "Factual (CF0)": "event_tp_era5_hourly_zarr_CF0_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0",
    "CF rain −8%":   "event_tp_era5_hourly_zarr_CF-8_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0",
    "CF SLR −0.1 m": "event_tp_era5_hourly_zarr_CF0_GTSMv41_CF-0.1_era5_hourly_spw_IBTrACS_CF0",
}

FLOOD_THRESHOLD = 0.05  # m
HMAX_VMAX       = 3.0   # m — upper limit of colormap
# ─────────────────────────────────────────────────────────────────────────────

SFINCS_ROOT = BASE / REGION / EVENT / "sfincs"
REGION_GEO  = SFINCS_ROOT / list(SCENARIOS.values())[0] / "gis" / "region.geojson"

for label, folder in SCENARIOS.items():
    p = SFINCS_ROOT / folder / "plot_output" / "sfincs_output_hmax_AllTime.tif"
    status = "✓" if p.exists() else "⚠️  NOT FOUND"
    print(f"{status}  {label}")

## Loading and aligning the flood maps

All scenarios use the same SFINCS grid, so the rasters are already aligned (same resolution, same extent). We load them into a dict for easy iteration.

In [ ]:
def load_hmax(folder: str) -> "xr.DataArray":
    path = SFINCS_ROOT / folder / "plot_output" / "sfincs_output_hmax_AllTime.tif"
    da = rxr.open_rasterio(path, masked=True).squeeze()
    da = da.where(da > FLOOD_THRESHOLD)  # mask dry / very-shallow cells
    return da

hmax_data = {label: load_hmax(folder) for label, folder in SCENARIOS.items()}

# Quick sanity check
for label, da in hmax_data.items():
    print(f"{label:25s} | max depth: {float(da.max()):.2f} m | shape: {da.shape}")

## Flood metrics per scenario

We define three inline helper functions so the computation is transparent.

In [ ]:
def pixel_area_m2(da) -> float:
    """Approximate pixel area [m²] using lat-adjusted degree-to-metre conversion."""
    res = abs(float(da.rio.resolution()[0]))
    lat = float(da.y.mean())
    return (res * 111_320) ** 2 * np.cos(np.deg2rad(lat))

def extent_km2(da) -> float:
    area = pixel_area_m2(da)
    return float((~np.isnan(da.values)).sum() * area / 1e6)

def volume_m3(da) -> float:
    area = pixel_area_m2(da)
    return float(np.nansum(da.values) * area)

def mean_depth_m(da) -> float:
    return float(np.nanmean(da.values))

# Compute for all scenarios
metrics = {}
for label, da in hmax_data.items():
    metrics[label] = {
        "extent_km2":  extent_km2(da),
        "volume_m3":   volume_m3(da),
        "mean_depth_m": mean_depth_m(da),
    }

# Pretty print
header = f"{'Scenario':25s} | {'Extent [km²]':>12} | {'Volume [Mm³]':>12} | {'Mean depth [m]':>14}"
print(header)
print("-" * len(header))
for label, m in metrics.items():
    print(f"{label:25s} | {m['extent_km2']:>12.1f} | {m['volume_m3']/1e6:>12.3f} | {m['mean_depth_m']:>14.3f}")

## Side-by-side flood maps

One panel per scenario. A shared colormap and vmax makes the visual difference immediately apparent.

In [ ]:
region = gpd.read_file(REGION_GEO)
labels = list(hmax_data.keys())
n_panels = len(labels)

fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 6), sharey=True)
if n_panels == 1:
    axes = [axes]

cmap = plt.cm.Blues

for ax, label in zip(axes, labels):
    da = hmax_data[label]
    im = da.plot(
        ax=ax, cmap=cmap, vmin=0.05, vmax=HMAX_VMAX,
        add_colorbar=False,
    )
    region.boundary.plot(ax=ax, color="black", linewidth=0.6, linestyle="--")
    ctx.add_basemap(ax, crs=da.rio.crs.to_string(),
                    source=ctx.providers.CartoDB.Positron, zoom=9)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")

# Shared colorbar
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=mcolors.Normalize(vmin=0.05, vmax=HMAX_VMAX))
fig.colorbar(sm, cax=cbar_ax, label="Max flood depth [m]")

fig.suptitle(f"TC {EVENT} — flood depth per scenario", fontsize=12, y=1.01)
plt.tight_layout()

## Attribution: difference maps

The attribution difference is `factual − counterfactual`: positive values (red) mean the factual flood was **deeper** than the counterfactual, i.e. climate change made the flood worse there. Negative values (blue) indicate cells where the counterfactual was actually deeper (e.g. due to spatial redistribution of flow).

We use a **diverging colormap** centred at zero.

In [ ]:
factual_label = labels[0]
factual_da    = hmax_data[factual_label].fillna(0)  # treat dry factual cells as 0

cf_labels = labels[1:]  # all counterfactuals
n_diff = len(cf_labels)

if n_diff == 0:
    print("Add at least one counterfactual to SCENARIOS to see attribution maps.")
else:
    fig, axes = plt.subplots(1, n_diff, figsize=(6 * n_diff, 6))
    if n_diff == 1:
        axes = [axes]

    diff_vmax = 1.5  # m — adjust if your region has deeper differences
    cmap_div  = plt.cm.RdBu_r

    for ax, cf_label in zip(axes, cf_labels):
        cf_da = hmax_data[cf_label].fillna(0)
        diff  = factual_da - cf_da  # positive → climate made it worse

        im = ax.imshow(
            diff.values,
            extent=[float(diff.x.min()), float(diff.x.max()),
                    float(diff.y.min()), float(diff.y.max())],
            origin="upper",
            cmap=cmap_div,
            vmin=-diff_vmax, vmax=diff_vmax,
        )
        region.boundary.plot(ax=ax, color="gray", linewidth=0.6)
        plt.colorbar(im, ax=ax, label="Δ depth [m]  (factual − CF)", shrink=0.7)
        ax.set_title(f"{factual_label}\n− {cf_label}", fontsize=9)

    fig.suptitle("Attribution: change in flood depth due to climate change", fontsize=11)
    plt.tight_layout()

## Attribution bar chart

Express the change in each metric as a percentage relative to the factual:  
`Δ% = (factual − counterfactual) / factual × 100`

A **positive** percentage means climate change increased that metric.

In [ ]:
if len(cf_labels) == 0:
    print("No counterfactuals to plot.")
else:
    metric_keys   = ["extent_km2", "volume_m3", "mean_depth_m"]
    metric_labels = ["Flood extent [km²]", "Flood volume [Mm³]", "Mean depth [m]"]
    factual_vals  = {k: metrics[factual_label][k] for k in metric_keys}

    x = np.arange(len(cf_labels))
    width = 0.22
    colors = ["#2196F3", "#4CAF50", "#FF9800"]

    fig, ax = plt.subplots(figsize=(7, 4))

    for i, (mk, ml, col) in enumerate(zip(metric_keys, metric_labels, colors)):
        pct_change = [
            (factual_vals[mk] - metrics[cf][mk]) / factual_vals[mk] * 100
            for cf in cf_labels
        ]
        bars = ax.bar(x + i * width, pct_change, width=width, label=ml, color=col, alpha=0.85)

        # Annotate bars
        for bar, val in zip(bars, pct_change):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + (0.3 if val >= 0 else -1.0),
                f"{val:+.1f}%", ha="center", va="bottom", fontsize=8
            )

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x + width)
    ax.set_xticklabels(cf_labels, rotation=15, ha="right", fontsize=9)
    ax.set_ylabel("Change relative to factual [%]")
    ax.set_title(f"TC {EVENT} — flood attribution (factual − counterfactual)")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

## Interpreting the results

- A **large positive percentage** (e.g., +30% in extent) means climate change was a major driver of the event's severity at that metric.
- **Small percentages** (< 5%) suggest that particular driver had a limited influence.
- To isolate the **compound effect**, compare the single-driver counterfactuals (CF rain only, CF SLR only) against the combined counterfactual (CF rain + CF SLR). Compound effects are non-additive in general.

For a full discussion of the attribution methodology, see the project report and Aleksandrova et al. (in prep.).

---

## Next steps

- **`04_damage_analysis.ipynb`** — translate the flood maps into building-level economic damage using FIAT outputs.
- For per-event attribution CSVs and published figures, see `postprocessing/attribution/flood_attribution.py`.